# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irssaa29/Machine-learning_01/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (content_hash_id), aggregated over March 2026 (month=2026-03), split into two halves: March 1–15 ("before") and March 16–31 ("after"). I start from the daily-grain table fact_content_daily_performance and roll it up per content_hash_id.

In [5]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/irssaa29/Machine-learning_01"
REPO_DIR = "Machine-learning_01"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/"
)
print(df_march.shape)
print("Date span:", df_march["report_date"].min(), "to", df_march["report_date"].max())

(9841378, 30)
Date span: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label: built from gsc_clicks, comparing March 1–15 sum vs March 16–31 sum — declining if the second half is lower.
Features (5, all GSC-only, first-half of March):
h1_impressions — total GSC impressions
h1_avg_position — mean GSC average position
h1_clicks — total GSC clicks
h1_ctr — derived: h1_clicks / h1_impressions (click-through rate)
h1_active_days — count of days with any GSC impressions (consistency of visibility)
Context: content_hash_id, client_hash_id, report_date, gsc_data_available.
Excluded: all ai_* referrer columns (sparse), and all ga4_*/sessions_* columns (only 21.7% coverage in this slice — dropped after checking availability in Section 3, to avoid building features that are mostly missing-not-zero).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three checks: (1) grain — one row per content_hash_id after aggregation, (2) row count + date span, (3) availability — filter both gsc_data_available and ga4_data_available to True and report survival.

In [6]:
# --- Split March into two halves ---
df_march["report_date"] = pd.to_datetime(df_march["report_date"])
first_half = df_march[df_march["report_date"].dt.day <= 15]
second_half = df_march[df_march["report_date"].dt.day > 15]

# --- Aggregate first half into GSC-only features ---
feat = first_half.groupby("content_hash_id").agg(
    h1_impressions=("gsc_impressions", "sum"),
    h1_avg_position=("gsc_avg_position", "mean"),
    h1_clicks=("gsc_clicks", "sum"),
    h1_active_days=("report_date", "nunique"),
).reset_index()

feat["h1_ctr"] = feat["h1_clicks"] / feat["h1_impressions"].replace(0, pd.NA)

# --- Aggregate second half for the label ---
h2 = second_half.groupby("content_hash_id").agg(h2_clicks=("gsc_clicks", "sum")).reset_index()

df_pair = feat.merge(h2, on="content_hash_id", how="inner")
df_pair["is_declining"] = (df_pair["h2_clicks"] < df_pair["h1_clicks"]).astype(int)

print(df_pair.shape)
print("Declining rate:", round(df_pair["is_declining"].mean(), 3))
df_pair.head()

# Check 1: grain
grain_check = df_pair.groupby("content_hash_id").size()
print("Grain check — max rows per content_hash_id (should be 1):", grain_check.max())

# Check 2: row count + date span
print("Row count after aggregation:", len(df_pair))
print("Date span:", df_march["report_date"].min(), "to", df_march["report_date"].max())

# Check 3: availability — filter both GSC and GA4 availability
avail = df_march.groupby("content_hash_id").agg(
    gsc_ok=("gsc_data_available", "any"),
    ga4_ok=("ga4_data_available", "any"),
).reset_index()

both_available = avail[(avail["gsc_ok"] == True) & (avail["ga4_ok"] == True)]
print(f"Pages with BOTH gsc_data_available and ga4_data_available True: {len(both_available)} out of {len(avail)}")
print(f"That's {round(len(both_available)/len(avail)*100, 1)}% of pages")

(319758, 8)
Declining rate: 0.091
Grain check — max rows per content_hash_id (should be 1): 1
Row count after aggregation: 319758
Date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
Pages with BOTH gsc_data_available and ga4_data_available True: 71888 out of 331437
That's 21.7% of pages


Five features, each knowable before the March 16–31 outcome exists:
1. h1_impressions — knowable because measured entirely within March 1–15, before the second-half period begins.
2. h1_avg_position — knowable because it's the average GSC position across March 1–15 only.
3. h1_clicks — knowable because summed only over March 1–15.
4. h1_ctr — knowable because derived purely from h1_clicks / h1_impressions, both confined to the first half.
5. h1_active_days — knowable because it counts distinct dates with GSC data in March 1–15 only.

The trap: adding one label-derived column on purpose to show the score jump, then removing it.
clicks_change is h2_clicks - h1_clicks, and is_declining is defined as h2_clicks < h1_clicks — so clicks_change is mathematically the label in disguise, which is why precision jumps to a near-perfect score below.

In [7]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

features_honest = ["h1_impressions", "h1_avg_position", "h1_clicks", "h1_ctr", "h1_active_days"]
X_honest = df_pair[features_honest].fillna(0)
y = df_pair["is_declining"].values

tree_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_honest.fit(X_honest, y)
score_honest = tree_honest.predict_proba(X_honest)[:, 1]
print("HONEST Precision@50:", round(precision_at_k(score_honest, y, 50), 3))

# --- Now deliberately leak: add a label-derived column ---
df_pair["clicks_change"] = df_pair["h2_clicks"] - df_pair["h1_clicks"]  # <- this IS the label, in disguise

features_leaky = features_honest + ["clicks_change"]
X_leaky = df_pair[features_leaky].fillna(0)

tree_leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_leaky.fit(X_leaky, y)
score_leaky = tree_leaky.predict_proba(X_leaky)[:, 1]
print("LEAKY Precision@50:", round(precision_at_k(score_leaky, y, 50), 3), " <- looks amazing, but it's cheating")

# --- Delete the leak, keep only the honest result ---
df_pair = df_pair.drop(columns=["clicks_change"])
print("\nFinal honest result (leak removed):", round(precision_at_k(score_honest, y, 50), 3))


/tmp/ipykernel_1914/1261390114.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_honest = df_pair[features_honest].fillna(0)


HONEST Precision@50: 0.66


/tmp/ipykernel_1914/1261390114.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_leaky = df_pair[features_leaky].fillna(0)


LEAKY Precision@50: 1.0  <- looks amazing, but it's cheating

Final honest result (leak removed): 0.66


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice has two real limitations. First, GA4 coverage is sparse: only 21.7% of pages had both GSC and GA4 data available in March, which is why all features here are GSC-only — a broader analysis would need to account for this gap rather than treat missing GA4 data as zero engagement. Second, this label only captures a two-week-over-two-week comparison within a single month; it cannot tell us whether a page's decline is a durable trend or short-term noise, since I have no visibility into performance before March or after.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.